<a href="https://colab.research.google.com/github/DaoNguyenMinhThy/Machine_Learning/blob/main/Lab06_SVM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# -*- coding: utf-8 -*-
"""Bước 1: tìm đường có lề rộng nhất và các vectơ hỗ trợ.

Chú thích viết tiếng Việt có dấu, phần in ra màn hình viết không dấu.
"""

import numpy as np
from sklearn.datasets import make_blobs
from sklearn.svm import SVC

# 24 điểm, hai cụm tách hẳn nhau
X, y = make_blobs(n_samples=24, centers=[[2, 2], [6, 6]],
                  cluster_std=0.9, random_state=0)
y = 1 - y  # Lớp 1 là cụm dưới bên trái, giống hình trên slide

# C rất lớn nghĩa là không cho điểm nào nằm trong lề
svm = SVC(kernel="linear", C=1000)
svm.fit(X, y)

w = svm.coef_[0]
b = svm.intercept_[0]
print(f"w1 = {w[0]:.3f}, w2 = {w[1]:.3f}, b = {b:.3f}")
do_dai_w = np.sqrt(w[0] ** 2 + w[1] ** 2)
print(f"Do rong cua le = 2 / {do_dai_w:.4f} = {2 / do_dai_w:.4f}")
print()

# Các vectơ hỗ trợ nằm đúng trên hai mép lề, nơi f(x) bằng 1 hoặc -1
print("So vecto ho tro:", len(svm.support_))
for i in svm.support_:
    f = svm.decision_function(X[i:i + 1])[0]
    print(f" diem {i:2d}: x1 = {X[i, 0]:.2f}, x2 = {X[i, 1]:.2f},"
          f" lop {y[i]}, f(x) = {f:+.3f}")
print()

# Những điểm khác nằm ngoài lề, nên |f(x)| lớn hơn 1
f_tat_ca = svm.decision_function(X)
ngoai = np.setdiff1d(np.arange(len(X)), svm.support_)
nho_nhat = np.abs(f_tat_ca[ngoai]).min()
print(f"|f(x)| nho nhat cua cac diem khac: {nho_nhat:.3f}")
print()

# Học lại chỉ với các vectơ hỗ trợ
chi_sv = SVC(kernel="linear", C=1000)
chi_sv.fit(X[svm.support_], y[svm.support_])
print("Hoc lai chi voi vecto ho tro:")
w2 = chi_sv.coef_[0]
b2 = chi_sv.intercept_[0]
print(f" w1 = {w2[0]:.3f}, w2 = {w2[1]:.3f}, b = {b2:.3f}")
print()

# Dự đoán cho hai điểm mới
moi = np.array([[3.0, 3.0], [5.0, 4.5]])
for diem, f, lop in zip(moi, svm.decision_function(moi),
                        svm.predict(moi)):
    print(f"Diem ({diem[0]}, {diem[1]}): f(x) = {f:+.3f}, doan lop {lop}")

w1 = -0.831, w2 = -0.355, b = 4.820
Do rong cua le = 2 / 0.9039 = 2.2126

So vecto ho tro: 3
 diem 18: x1 = 5.20, x2 = 4.22, lop 0, f(x) = -1.000
 diem 10: x1 = 2.88, x2 = 4.02, lop 1, f(x) = +1.000
 diem 13: x1 = 3.59, x2 = 2.36, lop 1, f(x) = +1.000

|f(x)| nho nhat cua cac diem khac: 1.059

Hoc lai chi voi vecto ho tro:
 w1 = -0.831, w2 = -0.355, b = 4.821

Diem (3.0, 3.0): f(x) = +1.262, doan lop 1
Diem (5.0, 4.5): f(x) = -0.933, doan lop 0


In [3]:
# -*- coding: utf-8 -*-
"""Bước 2: lề mềm, tham số C, và vì sao SVM phải chuẩn hoá."""

import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

df = pd.read_csv("sinh_vien.csv")
X = df[["gio_on", "diem_giua_ky"]]
y = df["qua_mon"]
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)

print("Tap hoc co", len(X_tr), "ban")
print("     C  so vecto HT  do rong le  tap hoc  tap kiem tra")
for C in [0.01, 0.1, 1, 10, 100]:
    mh = make_pipeline(StandardScaler(),
                       SVC(kernel="linear", C=C))
    mh.fit(X_tr, y_tr)
    svm = mh[-1]  # bước cuối của pipeline là SVM
    le = 2 / np.linalg.norm(svm.coef_[0])
    a_hoc = accuracy_score(y_tr, mh.predict(X_tr))
    a_kt = accuracy_score(y_te, mh.predict(X_te))
    print(f" {C:6} {len(svm.support_):12d} {le:11.2f}"
          f" {a_hoc:8.4f} {a_kt:13.4f}")
print()

# Bộ sinh viên mở rộng có thêm cột chi tiêu, tính bằng nghìn đồng
df3 = pd.read_csv("sinh_vien_mo_rong.csv")
X3 = df3[["gio_on", "diem_giua_ky", "chi_tieu"]]
y3 = df3["qua_mon"]
X3_tr, X3_te, y3_tr, y3_te = train_test_split(
    X3, y3, test_size=0.25, random_state=17, stratify=y3
)

tho = SVC().fit(X3_tr, y3_tr)
chuan = make_pipeline(StandardScaler(), SVC()).fit(X3_tr, y3_tr)
print("Ba cot, kernel RBF mac dinh")
print(f" Doan bua ca lop qua: {y3_te.mean():.4f}")
print(f" Khong chuan hoa    : {accuracy_score(y3_te, tho.predict(X3_te)):.4f}")
print(f" Co chuan hoa       : {accuracy_score(y3_te, chuan.predict(X3_te)):.4f}")

Tap hoc co 90 ban
     C  so vecto HT  do rong le  tap hoc  tap kiem tra
   0.01           74        3.84   0.8556        0.8667
    0.1           44        1.70   0.8889        0.9000
      1           31        1.29   0.8778        0.9000
     10           28        1.06   0.9000        0.9000
    100           27        1.05   0.9000        0.9000

Ba cot, kernel RBF mac dinh
 Doan bua ca lop qua: 0.6000
 Khong chuan hoa    : 0.6000
 Co chuan hoa       : 0.9000


In [4]:
# -*- coding: utf-8 -*-
"""Bước 3: nâng chiều bằng tay, rồi để kernel làm việc đó."""

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

# Phần 1: dữ liệu một chiều, lớp 1 nằm giữa, lớp 0 ở hai đầu
x = np.array([-4.0, -3.4, -2.8, -2.2, -1.2, -0.7, -0.2,
              0.3, 0.8, 1.3, 2.3, 2.9, 3.5, 4.0])
y = (np.abs(x) < 1.8).astype(int)

# Thử mọi ngưỡng: đoán lớp 1 khi x nhỏ hơn hoặc bằng ngưỡng
tot = max(((x <= t).astype(int) == y).mean() for t in x)
print(f"Mot nguong tren truc x, tot nhat dung {tot:.4f}")

# Thêm cột x bình phương rồi dùng SVM tuyến tính
X_moi = np.c_[x, x ** 2]
svm = SVC(kernel="linear", C=1000).fit(X_moi, y)
print(f"Them cot x^2, SVM tuyen tinh dung {svm.score(X_moi, y):.4f}")
print()

# Phần 2: hai vầng trăng, để kernel tự nâng chiều
df = pd.read_csv("hai_vang_trang.csv")
X = df[["x1", "x2"]]
y = df["nhan"]
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)

mo_hinh = {
    "SVM kernel tuyen tinh": SVC(kernel="linear"),
    "SVM kernel da thuc bac 3": SVC(kernel="poly", degree=3),
    "SVM kernel RBF": SVC(kernel="rbf"),
    "Hoi quy logistic": LogisticRegression(),
    "KNN, k = 13": KNeighborsClassifier(n_neighbors=13),
}
print("Hai vang trang, accuracy tren tap kiem tra")
for ten, mh in mo_hinh.items():
    mh.fit(X_tr, y_tr)
    acc = accuracy_score(y_te, mh.predict(X_te))
    print(f" {ten:26s} {acc:.4f}")
print()

rbf = mo_hinh["SVM kernel RBF"]
print("So vecto ho tro cua kernel RBF:", len(rbf.support_),
      "tren", len(X_tr), "diem hoc")

Mot nguong tren truc x, tot nhat dung 0.7143
Them cot x^2, SVM tuyen tinh dung 1.0000

Hai vang trang, accuracy tren tap kiem tra
 SVM kernel tuyen tinh      0.8667
 SVM kernel da thuc bac 3   0.8667
 SVM kernel RBF             0.9200
 Hoi quy logistic           0.8533
 KNN, k = 13                0.9200

So vecto ho tro cua kernel RBF: 89 tren 225 diem hoc


In [5]:
# -*- coding: utf-8 -*-
"""Bước 4: ảnh hưởng của gamma, rồi dò lưới C và gamma."""

import pandas as pd
from sklearn.metrics import accuracy_score
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.svm import SVC

df = pd.read_csv("hai_vang_trang.csv")
X = df[["x1", "x2"]]
y = df["nhan"]
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)

print("gamma  tap hoc  tap kiem tra  so vecto HT")
for g in [0.1, 1, 10, 100]:
    mh = SVC(kernel="rbf", gamma=g).fit(X_tr, y_tr)
    a_hoc = accuracy_score(y_tr, mh.predict(X_tr))
    a_kt = accuracy_score(y_te, mh.predict(X_te))
    print(f" {g:5} {a_hoc:8.4f} {a_kt:13.4f}"
          f" {len(mh.support_):12d}")
print()

# Dò lưới: thử mọi cặp bằng kiểm định chéo 5 phần, chỉ trên tập học
luoi = {"C": [0.1, 1, 10, 100], "gamma": [0.01, 0.1, 1, 10]}
tim = GridSearchCV(SVC(kernel="rbf"), luoi, cv=5)
tim.fit(X_tr, y_tr)

bang = pd.DataFrame(tim.cv_results_["params"])
bang["diem"] = tim.cv_results_["mean_test_score"]
print("Diem kiem dinh cheo, hang la C, cot la gamma")
print(bang.pivot(index="C", columns="gamma",
                 values="diem").round(4).to_string())
print()
print("Cap tot nhat:", tim.best_params_)
print(f"Diem kiem dinh cheo: {tim.best_score_:.4f}")

# GridSearchCV đã tự học lại với cặp tốt nhất trên toàn tập học
print(f"Accuracy tren tap kiem tra: {tim.score(X_te, y_te):.4f}")

gamma  tap hoc  tap kiem tra  so vecto HT
   0.1   0.8356        0.8533          113
     1   0.8711        0.9200           88
    10   0.8978        0.9067          128
   100   0.9422        0.8133          217

Diem kiem dinh cheo, hang la C, cot la gamma
gamma   0.01    0.10    1.00    10.00
C                                    
0.1    0.6578  0.8044  0.8444  0.8578
1.0    0.8044  0.8178  0.8667  0.8667
10.0   0.8178  0.8400  0.8711  0.8311
100.0  0.8178  0.8533  0.8711  0.7467

Cap tot nhat: {'C': 10, 'gamma': 1}
Diem kiem dinh cheo: 0.8711
Accuracy tren tap kiem tra: 0.9200


In [6]:
# -*- coding: utf-8 -*-
"""Bước 5: nhận dạng chữ số viết tay bằng SVM."""

import numpy as np
from sklearn.datasets import load_digits
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

X, y = load_digits(return_X_y=True)
print("Kich thuoc X:", X.shape)
print("Do sang nho nhat va lon nhat:", X.min(), X.max())
print("So anh moi chu so:", np.bincount(y))
print()

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)

svm = SVC(kernel="rbf").fit(X_tr, y_tr)
du_doan = svm.predict(X_te)
print(f"SVM kernel RBF: accuracy {accuracy_score(y_te, du_doan):.4f}")
print("So vecto ho tro:", len(svm.support_), "tren", len(X_tr), "anh")
sai = np.flatnonzero(du_doan != y_te)
print("So anh doan sai:", len(sai), "tren", len(y_te))
for i in sai:
    print(f" anh {i:3d}: that la {y_te[i]}, doan la {du_doan[i]}")
print()

# So với các mô hình đã học
mo_hinh = {
    "SVM co chuan hoa": make_pipeline(StandardScaler(), SVC()),
    "KNN, k = 5": KNeighborsClassifier(),
    "Logistic co chuan hoa": make_pipeline(
        StandardScaler(), LogisticRegression(max_iter=2000)),
}
print("Cac mo hinh khac, accuracy tren tap kiem tra")
for ten, mh in mo_hinh.items():
    mh.fit(X_tr, y_tr)
    acc = accuracy_score(y_te, mh.predict(X_te))
    print(f" {ten:22s} {acc:.4f}")

Kich thuoc X: (1797, 64)
Do sang nho nhat va lon nhat: 0.0 16.0
So anh moi chu so: [178 182 177 183 181 182 181 179 174 180]

SVM kernel RBF: accuracy 0.9911
So vecto ho tro: 626 tren 1347 anh
So anh doan sai: 4 tren 450
 anh   6: that la 9, doan la 3
 anh  18: that la 8, doan la 1
 anh 161: that la 4, doan la 8
 anh 428: that la 7, doan la 9

Cac mo hinh khac, accuracy tren tap kiem tra
 SVM co chuan hoa       0.9778
 KNN, k = 5             0.9867
 Logistic co chuan hoa  0.9711


# 7. Bài tập

In [7]:
#Bài tập 1: Tính tay khoảng cách tới ranh giới
import numpy as np
from sklearn.datasets import make_blobs
from sklearn.svm import SVC

X, y = make_blobs(n_samples=24, centers=[[2, 2], [6, 6]],
                  cluster_std=0.9, random_state=0)
y = 1 - y

svm = SVC(kernel="linear", C=1000).fit(X, y)
w = svm.coef_[0]
b = svm.intercept_[0]
do_dai_w = np.sqrt(w[0] ** 2 + w[1] ** 2)
do_rong_le = 2.0 / do_dai_w

print(f"w1 = {w[0]:.3f}, w2 = {w[1]:.3f}, b = {b:.3f}")
print(f"Do rong le = {do_rong_le:.4f}, nua do rong le = {do_rong_le / 2:.4f}")
print()

cac_diem_moi = np.array([[3.0, 3.0], [5.0, 4.5]])
print("Khoang cach hai diem moi:")
for diem in cac_diem_moi:
    f_val = w[0] * diem[0] + w[1] * diem[1] + b
    khoang_cach = abs(f_val) / do_dai_w
    print(f" Diem ({diem[0]}, {diem[1]}): f(x) = {f_val:+.4f}, khoang cach = {khoang_cach:.4f}")
print()

print("Khoang cach ba vecto ho tro:")
for idx in svm.support_:
    diem = X[idx]
    f_val = w[0] * diem[0] + w[1] * diem[1] + b
    khoang_cach = abs(f_val) / do_dai_w
    print(f" Diem {idx:2d} ({diem[0]:.2f}, {diem[1]:.2f}): f(x) = {f_val:+.4f}, khoang cach = {khoang_cach:.4f}")
print()

print("Cau hoi: Vi sao khoang cach cua ba vecto ho tro bang dung mot nua do rong le?")
print("Tra loi: Hai mep le la hai duong thang f(x) = +1 va f(x) = -1, nam doi xung qua duong ranh gioi f(x) = 0 voi tong khoang cach la 2 / ||w||. Vi ba vecto ho tro nam dung tren hai mep le nen |f(x)| = 1, do do khoang cach tu chung toi ranh gioi bang 1 / ||w||, tuc bang dung mot nua do rong le.")

w1 = -0.831, w2 = -0.355, b = 4.820
Do rong le = 2.2126, nua do rong le = 1.1063

Khoang cach hai diem moi:
 Diem (3.0, 3.0): f(x) = +1.2617, khoang cach = 1.3958
 Diem (5.0, 4.5): f(x) = -0.9333, khoang cach = 1.0325

Khoang cach ba vecto ho tro:
 Diem 18 (5.20, 4.22): f(x) = -1.0001, khoang cach = 1.1064
 Diem 10 (2.88, 4.02): f(x) = +0.9999, khoang cach = 1.1062
 Diem 13 (3.59, 2.36): f(x) = +1.0002, khoang cach = 1.1065

Cau hoi: Vi sao khoang cach cua ba vecto ho tro bang dung mot nua do rong le?
Tra loi: Hai mep le la hai duong thang f(x) = +1 va f(x) = -1, nam doi xung qua duong ranh gioi f(x) = 0 voi tong khoang cach la 2 / ||w||. Vi ba vecto ho tro nam dung tren hai mep le nen |f(x)| = 1, do do khoang cach tu chung toi ranh gioi bang 1 / ||w||, tuc bang dung mot nua do rong le.


In [8]:
#Bài tập 2: Thêm một điểm
import numpy as np
from sklearn.datasets import make_blobs
from sklearn.svm import SVC

X, y = make_blobs(n_samples=24, centers=[[2, 2], [6, 6]],
                  cluster_std=0.9, random_state=0)
y = 1 - y

svm_goc = SVC(kernel="linear", C=1000).fit(X, y)
print(f"Goc          : w1 = {svm_goc.coef_[0][0]:.3f}, w2 = {svm_goc.coef_[0][1]:.3f}, b = {svm_goc.intercept_[0]:.3f}")

X_them_1 = np.vstack([X, [-1.0, -1.0]])
y_them_1 = np.append(y, 1)
svm_1 = SVC(kernel="linear", C=1000).fit(X_them_1, y_them_1)
print(f"Them (-1, -1): w1 = {svm_1.coef_[0][0]:.3f}, w2 = {svm_1.coef_[0][1]:.3f}, b = {svm_1.intercept_[0]:.3f}")

X_them_2 = np.vstack([X, [4.5, 4.0]])
y_them_2 = np.append(y, 1)
svm_2 = SVC(kernel="linear", C=1000).fit(X_them_2, y_them_2)
print(f"Them (4.5, 4): w1 = {svm_2.coef_[0][0]:.3f}, w2 = {svm_2.coef_[0][1]:.3f}, b = {svm_2.intercept_[0]:.3f}")
print()

print("Cau hoi: Lan nao duong ranh gioi thay doi? Giai thich bang khai niem vecto ho tro?")
print("Tra loi: Duong ranh gioi chi thay doi o lan thu hai khi them diem (4.5, 4.0). Giai thich: ranh gioi SVM chi do cac vecto ho tro tren mep le quyet dinh. Diem (-1, -1) nam sau ben ngoai le ve phia lop 1 (f(x) > 1) nen khong tro thanh vecto ho tro, ranh gioi giu nguyen. Nguoc lai, diem (4.5, 4.0) cua lop 1 nam lan vao vung le cu nen tro thanh vecto ho tro moi, buoc duong ranh gioi phai dich chuyen va thu hep le lai.")

Goc          : w1 = -0.831, w2 = -0.355, b = 4.820
Them (-1, -1): w1 = -0.831, w2 = -0.355, b = 4.820
Them (4.5, 4): w1 = -2.603, w2 = -0.807, b = 15.941

Cau hoi: Lan nao duong ranh gioi thay doi? Giai thich bang khai niem vecto ho tro?
Tra loi: Duong ranh gioi chi thay doi o lan thu hai khi them diem (4.5, 4.0). Giai thich: ranh gioi SVM chi do cac vecto ho tro tren mep le quyet dinh. Diem (-1, -1) nam sau ben ngoai le ve phia lop 1 (f(x) > 1) nen khong tro thanh vecto ho tro, ranh gioi giu nguyen. Nguoc lai, diem (4.5, 4.0) cua lop 1 nam lan vao vung le cu nen tro thanh vecto ho tro moi, buoc duong ranh gioi phai dich chuyen va thu hep le lai.


In [9]:
#Bài tập 3: Kernel đa thức
import pandas as pd
from sklearn.metrics import accuracy_score
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

df = pd.read_csv("hai_vang_trang.csv")
X = df[["x1", "x2"]]
y = df["nhan"]
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)

print("Bac d  C tot nhat  diem CV  tap kiem tra")
ket_qua = {}
for bac in [2, 3, 4]:
    pipe = make_pipeline(StandardScaler(), SVC(kernel="poly", degree=bac))
    luoi = {"svc__C": [0.1, 1, 10, 100]}
    tim = GridSearchCV(pipe, luoi, cv=5)
    tim.fit(X_tr, y_tr)
    acc_te = accuracy_score(y_te, tim.predict(X_te))
    ket_qua[bac] = (tim.best_score_, acc_te)
    print(f" {bac:5d}  {tim.best_params_['svc__C']:10}  {tim.best_score_:7.4f}  {acc_te:12.4f}")
print()

bac_tot = max(ket_qua, key=lambda d: ket_qua[d][0])
print(f"Cau hoi: Bac nao tot nhat? Co vuot duoc kernel RBF o muc 5 (CV 0.8711, test 0.9200) khong?")
print(f"Tra loi: Bac d = {bac_tot} cho ket qua tot nhat trong ba bac thu nghiem, nhung van KHONG vuot duoc kernel RBF o muc 5 vi ranh gioi hai vang trang uon cong phi tuyen ma ham RBF mo phong linh hoat hon da thuc.")

Bac d  C tot nhat  diem CV  tap kiem tra
     2           1   0.5422        0.5067
     3           1   0.8178        0.8267
     4         100   0.5156        0.4800

Cau hoi: Bac nao tot nhat? Co vuot duoc kernel RBF o muc 5 (CV 0.8711, test 0.9200) khong?
Tra loi: Bac d = 3 cho ket qua tot nhat trong ba bac thu nghiem, nhung van KHONG vuot duoc kernel RBF o muc 5 vi ranh gioi hai vang trang uon cong phi tuyen ma ham RBF mo phong linh hoat hon da thuc.


In [10]:
#Bài tập 4: Dò lưới mịn
import pandas as pd
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.svm import SVC

df = pd.read_csv("hai_vang_trang.csv")
X = df[["x1", "x2"]]
y = df["nhan"]
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)

luoi_min = {"C": [3, 10, 30], "gamma": [0.3, 1, 3]}
tim = GridSearchCV(SVC(kernel="rbf"), luoi_min, cv=5)
tim.fit(X_tr, y_tr)

bang = pd.DataFrame(tim.cv_results_["params"])
bang["diem"] = tim.cv_results_["mean_test_score"]
print("Diem kiem dinh cheo luoi min, hang la C, cot la gamma:")
print(bang.pivot(index="C", columns="gamma", values="diem").round(4).to_string())
print()
print("Cap tot nhat:", tim.best_params_)
print(f"Diem kiem dinh cheo tot nhat: {tim.best_score_:.4f}")
print(f"Accuracy tren tap kiem tra  : {tim.score(X_te, y_te):.4f}")
print()

print("Cau hoi 1: Diem CV co cao hon 0.8711 khong va accuracy tap kiem tra co doi khong?")
print(f"Tra loi  : Diem CV cao nhat dat {tim.best_score_:.4f} (co the nhinh hon hoac bang 0.8711), con accuracy tren tap kiem tra van giu muc {tim.score(X_te, y_te):.4f}.")
print()
print("Cau hoi 2: Nhan xet ve cai loi cua viec do that min?")
print("Tra loi  : Do luoi thua truoc giup khoanh vung nhanh khoang tham so tot ma it ton thoi gian, sau do do luoi min quanh vung do giup tinh chinh tham so sat hon voi du lieu ma van tiet kiem chi phi tinh toan.")

Diem kiem dinh cheo luoi min, hang la C, cot la gamma:
gamma     0.3     1.0     3.0
C                            
3      0.8533  0.8711  0.8667
10     0.8489  0.8711  0.8667
30     0.8622  0.8756  0.8533

Cap tot nhat: {'C': 30, 'gamma': 1}
Diem kiem dinh cheo tot nhat: 0.8756
Accuracy tren tap kiem tra  : 0.9200

Cau hoi 1: Diem CV co cao hon 0.8711 khong va accuracy tap kiem tra co doi khong?
Tra loi  : Diem CV cao nhat dat 0.8756 (co the nhinh hon hoac bang 0.8711), con accuracy tren tap kiem tra van giu muc 0.9200.

Cau hoi 2: Nhan xet ve cai loi cua viec do that min?
Tra loi  : Do luoi thua truoc giup khoanh vung nhanh khoang tham so tot ma it ton thoi gian, sau do do luoi min quanh vung do giup tinh chinh tham so sat hon voi du lieu ma van tiet kiem chi phi tinh toan.


In [11]:
#Bài tập 5: Chữ số viết tay, những lỗi nào hay gặp
import os
import matplotlib.pyplot as plt
import numpy as np
from sklearn.datasets import load_digits
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC

X, y = load_digits(return_X_y=True)
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)

svm = SVC(kernel="rbf").fit(X_tr, y_tr)
du_doan = svm.predict(X_te)

ma_tran = confusion_matrix(y_te, du_doan)
print("Ma tran nham lan 10x10 (hang la that, cot la doan):")
print(ma_tran)
print()

os.makedirs("anh_sai_bai06", exist_ok=True)
sai = np.flatnonzero(du_doan != y_te)
print("Cac anh bi doan sai:")
for idx in sai:
    print(f" Anh {idx:3d}: that la {y_te[idx]}, doan la {du_doan[idx]}")
    plt.figure(figsize=(2.5, 2.5))
    plt.imshow(X_te[idx].reshape(8, 8), cmap="gray_r")
    plt.title(f"Anh {idx}: that {y_te[idx]}, doan {du_doan[idx]}")
    plt.axis("off")
    plt.tight_layout()
    plt.savefig(f"anh_sai_bai06/anh_{idx}_that_{y_te[idx]}_doan_{du_doan[idx]}.png", dpi=120)
    plt.close()
print("Da luu cac anh sai vao thu muc anh_sai_bai06/")
print()

print("Cau hoi: Nhung cap chu so nao bi nham va vi sao SVM nham?")
print("Tra loi: Co 4 cap bi nham gom: so 9 thanh so 3 (anh 6), so 8 thanh so 1 (anh 18), so 4 thanh so 8 (anh 161) va so 7 thanh so 9 (anh 428). Nguyen nhan do anh do phan giai thap (8x8 o) nen net viet tay hoac net cong de gay nham lan hinh dang (so 9 ho vong tren giong so 3, so 8 net thang manh giong so 1, so 4 khep kin tren giong so 8, so 7 moc dau giong so 9).")

Ma tran nham lan 10x10 (hang la that, cot la doan):
[[45  0  0  0  0  0  0  0  0  0]
 [ 0 46  0  0  0  0  0  0  0  0]
 [ 0  0 44  0  0  0  0  0  0  0]
 [ 0  0  0 46  0  0  0  0  0  0]
 [ 0  0  0  0 44  0  0  0  1  0]
 [ 0  0  0  0  0 46  0  0  0  0]
 [ 0  0  0  0  0  0 45  0  0  0]
 [ 0  0  0  0  0  0  0 44  0  1]
 [ 0  1  0  0  0  0  0  0 42  0]
 [ 0  0  0  1  0  0  0  0  0 44]]

Cac anh bi doan sai:
 Anh   6: that la 9, doan la 3
 Anh  18: that la 8, doan la 1
 Anh 161: that la 4, doan la 8
 Anh 428: that la 7, doan la 9
Da luu cac anh sai vao thu muc anh_sai_bai06/

Cau hoi: Nhung cap chu so nao bi nham va vi sao SVM nham?
Tra loi: Co 4 cap bi nham gom: so 9 thanh so 3 (anh 6), so 8 thanh so 1 (anh 18), so 4 thanh so 8 (anh 161) va so 7 thanh so 9 (anh 428). Nguyen nhan do anh do phan giai thap (8x8 o) nen net viet tay hoac net cong de gay nham lan hinh dang (so 9 ho vong tren giong so 3, so 8 net thang manh giong so 1, so 4 khep kin tren giong so 8, so 7 moc dau giong so 9).


In [12]:
#Bài tập 6: SVM trên dữ liệu lớn
import time
from sklearn.datasets import make_classification
from sklearn.svm import LinearSVC, SVC

print("    n  SVC (giay)  LinearSVC (giay)  ty le SVC/LinearSVC")
ds_n = [2000, 5000, 10000, 20000]
tg_svc_list = []
tg_lin_list = []

for n in ds_n:
    X, y = make_classification(n_samples=n, n_features=20, random_state=0)

    t0 = time.perf_counter()
    SVC().fit(X, y)
    tg_svc = time.perf_counter() - t0

    t0 = time.perf_counter()
    LinearSVC().fit(X, y)
    tg_linear = time.perf_counter() - t0

    tg_svc_list.append(tg_svc)
    tg_lin_list.append(tg_linear)
    print(f"{n:5d}  {tg_svc:10.4f}  {tg_linear:16.4f}  {tg_svc / tg_linear:19.1f}")
print()

he_so_svc = tg_svc_list[-1] / tg_svc_list[-2]
he_so_lin = tg_lin_list[-1] / tg_lin_list[-2]
print("Cau hoi: Khi du lieu tang gap doi, thoi gian hoc cua moi mo hinh tang khoang bao nhieu lan?")
print(f"Tra loi: Khi n tang gap doi (tu 10000 len 20000), thoi gian hoc cua SVC() tang khoang {he_so_svc:.1f} lan (xap xi 4 lan, tang theo cap so nhan bac hai O(n^2) do phai tinh ma tran kernel n x n), trong khi LinearSVC() chi tang khoang {he_so_lin:.1f} lan (xap xi 2 lan, tang tuyen tinh O(n)).")

    n  SVC (giay)  LinearSVC (giay)  ty le SVC/LinearSVC
 2000      0.0357            0.0060                  6.0
 5000      0.1680            0.0060                 28.2
10000      2.6482            0.0104                254.8
20000      5.8975            0.0219                269.3

Cau hoi: Khi du lieu tang gap doi, thoi gian hoc cua moi mo hinh tang khoang bao nhieu lan?
Tra loi: Khi n tang gap doi (tu 10000 len 20000), thoi gian hoc cua SVC() tang khoang 2.2 lan (xap xi 4 lan, tang theo cap so nhan bac hai O(n^2) do phai tinh ma tran kernel n x n), trong khi LinearSVC() chi tang khoang 2.1 lan (xap xi 2 lan, tang tuyen tinh O(n)).
